# 1-board: rest-noise

Measure the noise of each raw channel with the bridge off, so later notebooks know how much of what they see in one sample is the chain and not the motor.

## Purpose

Every current, terminal and rail reading the firmware acts on is one ADC sample per control tick. The spread of those samples with nothing driving is the floor under every estimate built on them: the current limit's trip, the stall detector, the winding thermometer's ratio, the back-EMF read on a coasting terminal. This experiment measures that spread per channel at the control rate, at two supply levels, and looks for structure in it, since a noise that repeats with the telemetry frame or at a line frequency is a coupling to fix rather than a floor to accept.

## Procedure

The board sits on the DPS with the bare load across its motor terminals, which draws nothing with the bridge off, and the adapter on its bus at 3 Mbps. `capture.py` powers the board, then at each of 5.0 V and 7.4 V records three torque-off streams of ten seconds and one more with the board NTC in the row in place of the position. Each stream is an `osc sweep` whose capture is its baseline; the 5 % rung of 50 ms that follows is the smallest the tool records and is left out of the analysis.

1. The board on the DPS, the bare load on the terminals, the adapter on the bus.
2. `capture.py`. It asks for the leads off the DPS before the output comes on, then on.

## Data

- `rest-<V>V-<k>/sweep.csv`, `meta.json`: one row per control tick. Segment 0 is the rest. `current_raw`, `vmotor_a`, `vmotor_b` and `vbus_raw` are ADC counts; `tick` is the control tick, and a jump in it is a dropped frame.
- `rest-<V>V-ntc/`: the same with `ntc_raw` in place of `pos`.

### One stream

The first rest at 7.4 V, every raw channel against time.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from oscnb import b41t, boards, display, laps, results, tel
from oscnb.display import show

BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#898781"
BOARD = boards.BOARDS[boards.DEFAULT]
CAPTURE = Path("../../../telemetry") / BOARD.key / "rest-noise" / "capture-1"
CAPTURE = CAPTURE.resolve()
print(BOARD.key, "from", CAPTURE)

In [ ]:
folders = sorted(p for p in CAPTURE.glob("rest-*") if (p / "sweep.csv").exists())
print(len(folders), "streams")
df, meta = tel.load(folders[0], BOARD)
rest = df[df["seg"] == 0]
fig, ax = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
for a, c in zip(ax, ["current_raw", "vmotor_a", "vmotor_b", "vbus_raw"]):
    a.plot(rest["t"], rest[c], lw=0.4, color=BLUE); a.set_ylabel(c); a.grid(alpha=0.3)
ax[-1].set_xlabel("time (s)")
plt.show()
print(f"{len(rest)} rows over {rest['t'].iloc[-1]:.1f} s; gaps: {tel.gaps(df).to_dict() or 'none'}")

Each channel at rest for ten seconds. The current channel sits on the amplifier's zero, the taps on the bias node, the rail on the supply.

## Method

The spread of each channel over the rest, as a standard deviation in counts, then in the units the chain converts to. Three streams per level give three values, and the spread between them says how stable the floor is. Two structures are checked: the mean of each channel by position in the 16-row telemetry frame, which shows a coupling from the bus, and the spectrum of the current channel, which shows a line or a switching frequency.

### Spread per channel

In [ ]:
rows = []
for p in folders:
    df, meta = tel.load(p, BOARD)
    rest = df[df["seg"] == 0]
    volts = float(p.name.split("-")[1].rstrip("V"))
    row = {"stream": p.name, "volts": volts, "rows": len(rest), "missing": int(tel.gaps(df).sum())}
    for c in tel.RAW:
        if c in rest and rest[c].notna().any():
            row[c + "_mean"] = rest[c].mean(); row[c + "_sd"] = rest[c].std()
    rows.append(row)
spread = pd.DataFrame(rows)
show(spread, n=None)

One row per stream: the mean and standard deviation of each channel in counts.

In [ ]:
sd = spread.groupby("volts")[[c + "_sd" for c in ["current_raw", "vmotor_a", "vmotor_b", "vbus_raw"]]].agg(["mean", "std"])
show(sd, n=None)
units = {"current_raw": ("mA", BOARD.a_per_count * 1e3), "vmotor_a": ("mV at the terminal", BOARD.v_term_per_count * 1e3),
         "vmotor_b": ("mV at the terminal", BOARD.v_term_per_count * 1e3), "vbus_raw": ("mV on the rail", BOARD.v_rail_per_count * 1e3)}
for c, (u, k) in units.items():
    s = spread[c + "_sd"].mean()
    print(f"{c:<12} {s:.2f} counts rms, {s * k:.1f} {u}")
ntc = spread["ntc_raw_sd"].dropna() if "ntc_raw_sd" in spread else pd.Series(dtype=float)
if len(ntc):
    print(f"ntc_raw      {ntc.mean():.2f} counts rms")

The standard deviation per channel at each level, with its spread over the repeats, then each one in the unit its chain reads.

### Structure

In [ ]:
df, meta = tel.load(folders[0], BOARD)
rest = df[df["seg"] == 0]
by = rest.groupby(rest["tick"] % 16)[["current_raw", "vmotor_a", "vmotor_b", "vbus_raw"]].mean()
fs = meta.get("tick_hz", 20000)
x = rest["current_raw"].to_numpy(float) - rest["current_raw"].mean()
f = np.fft.rfftfreq(len(x), 1 / fs); P = np.abs(np.fft.rfft(x)) ** 2 / len(x)
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
(by - by.mean()).plot(ax=ax[0], marker="o", lw=0.8); ax[0].set_xlabel("row in the telemetry frame"); ax[0].set_ylabel("mean less the stream mean (counts)"); ax[0].grid(alpha=0.3)
ax[1].semilogy(f[1:], P[1:], lw=0.5, color=BLUE); ax[1].set_xlabel("frequency (Hz)"); ax[1].set_ylabel("current channel power"); ax[1].grid(alpha=0.3)
plt.show()
peak = f[1:][np.argmax(P[1:])]
print(f"frame-position spread of the current channel {by['current_raw'].max() - by['current_raw'].min():.2f} counts; strongest line at {peak:.0f} Hz")

Left, each channel's mean by row in the telemetry frame, less the stream mean; a bus coupling shows as a pattern here. Right, the current channel's spectrum; a line or a switching supply shows as a peak over the floor.

## Results

The floor per channel goes into the result as counts rms, with the spread over repeats as the bounds, together with the frame-position spread of the current channel.

In [ ]:
def q(c):
    s = spread[c + "_sd"]
    return results.quantity(s.mean(), lo=s.min(), hi=s.max(), unit="counts rms")
results.write("1-board", "rest-noise", CAPTURE, pair=BOARD.key,
              notes="torque-off streams at the control rate, bridge off, bare load on the terminals",
              current_noise=q("current_raw"), tap_a_noise=q("vmotor_a"), tap_b_noise=q("vmotor_b"), rail_noise=q("vbus_raw"),
              **({"ntc_noise": q("ntc_raw")} if "ntc_raw_sd" in spread and spread["ntc_raw_sd"].notna().any() else {}),
              current_frame_spread=results.quantity(by["current_raw"].max() - by["current_raw"].min(), unit="counts"))

## Limitations

- The bridge is off, so this is the chain's floor with no switching. The floor under drive, with the bridge chopping and the current climbing inside every window, is a different number and comes from the sample-timing experiment's ladders.
- Ten seconds at 20 kHz resolves nothing under a few hertz. A slow wander of the zero shows in the laps of current-scale as a difference between their rests.
- The DPS is the supply. A pack's floor may differ by its own impedance, and USB power by its ripple.

## Open questions

- Whether the floor changes with the load on the terminals. The same streams with the terminals open would say what the load's lead loop picks up.

## Conclusion

To be written from the capture.